# Prepare Calendar Table

Create and validate the standard silver calendar table for CareSync, covering dates from 1940-01-01 through 2030-12-31 with common date dimension attributes for reporting and joins.

In [0]:
CREATE TABLE IF NOT EXISTS adb_caresync.silver.calendar AS
WITH dates AS (
  SELECT explode(sequence(to_date('1940-01-01'), to_date('2030-12-31'), interval 1 day)) AS calendar_date
)
SELECT
  CAST(date_format(calendar_date, 'yyyyMMdd') AS INT) AS datekey,
  calendar_date AS `date`,
  year(calendar_date) AS `year`,
  quarter(calendar_date) AS quarter,
  concat('Q', quarter(calendar_date)) AS quarter_name,
  trunc(calendar_date, 'QUARTER') AS quarter_start_date,
  date_sub(add_months(trunc(calendar_date, 'QUARTER'), 3), 1) AS quarter_end_date,
  month(calendar_date) AS `month`,
  date_format(calendar_date, 'MMMM') AS month_name,
  date_format(calendar_date, 'MMM') AS month_short_name,
  CAST(date_format(calendar_date, 'yyyyMM') AS INT) AS year_month,
  trunc(calendar_date, 'MONTH') AS month_start_date,
  last_day(calendar_date) AS month_end_date,
  weekofyear(calendar_date) AS `week`,
  date_sub(calendar_date, weekday(calendar_date)) AS week_start_date,
  date_add(date_sub(calendar_date, weekday(calendar_date)), 6) AS week_end_date,
  weekday(calendar_date) + 1 AS day_of_week,
  date_format(calendar_date, 'EEEE') AS day_name,
  date_format(calendar_date, 'EEE') AS day_short_name,
  dayofmonth(calendar_date) AS day_of_month,
  dayofyear(calendar_date) AS day_of_year,
  CASE WHEN weekday(calendar_date) >= 5 THEN true ELSE false END AS is_weekend,
  calendar_date = trunc(calendar_date, 'MONTH') AS is_month_start,
  calendar_date = last_day(calendar_date) AS is_month_end,
  calendar_date = trunc(calendar_date, 'QUARTER') AS is_quarter_start,
  calendar_date = date_sub(add_months(trunc(calendar_date, 'QUARTER'), 3), 1) AS is_quarter_end,
  calendar_date = trunc(calendar_date, 'YEAR') AS is_year_start,
  calendar_date = date_sub(add_months(trunc(calendar_date, 'YEAR'), 12), 1) AS is_year_end,
  trunc(calendar_date, 'YEAR') AS year_start_date,
  date_sub(add_months(trunc(calendar_date, 'YEAR'), 12), 1) AS year_end_date
FROM dates;

In [0]:
SELECT
  MIN(`date`) AS min_date,
  MAX(`date`) AS max_date,
  COUNT(*) AS row_count,
  datediff(to_date('2030-12-31'), to_date('1940-01-01')) + 1 AS expected_row_count,
  COUNT(*) = datediff(to_date('2030-12-31'), to_date('1940-01-01')) + 1 AS row_count_matches_expected
FROM adb_caresync.silver.calendar;

In [0]:
SELECT *
FROM adb_caresync.silver.calendar
WHERE year=2026
ORDER BY `date`
--LIMIT 5;